# Pipeline Integration Demo

This notebook demonstrates the end-to-end sidecar flow with draft-aware preprocessing and state estimation.

The goal is to show how to run a short local simulation and inspect emitted advisory actions.

In [ ]:
from rheed_core.main import build_pipeline

cfg = {
    "dummy": {
        "sample_rate_hz": 25.0,
        "frame_rate_hz": 3.0,
        "osc_period_s": 3.0,
        "noise_std": 0.12,
        "seed": 5,
    },
    "preprocess": {
        "trend_window": 64,
        "clip_sigma": 4.0,
        "median_kernel_size": 5,
        "fft_band": [0.05, 5.0],
        "sample_rate_hz": 25.0,
        "fft_window": 128,
    },
    "state": {
        "max_points": 400,
        "min_period_s": 0.2,
        "max_period_s": 15.0,
        "tau_peak_prominence": 0.05,
        "tau_min_points": 8,
    },
    "policy": {
        "advisory_mode": True,
        "drift_threshold": 7.0,
        "bad_frame_limit": 6,
        "period_cv_threshold": 0.2,
        "tau_cv_threshold": 0.35,
    },
    "logging": {
        "path": "logs/notebook_events.jsonl",
    },
}

pipeline, operator = build_pipeline(cfg)


In [ ]:
# Run a short simulation.
pipeline.run(duration_s=4.0, sleep_s=0.01)
print(f"Advisory actions emitted: {len(operator.actions)}")
if operator.actions:
    print("Last action:", operator.actions[-1].message)


## Next step for real data
- Replace `DummyCollector` with a TSST export stream adapter that yields `SignalPacket` and `FramePacket`.
- Keep the same `build_pipeline` shape and only swap the collector implementation.
- Use JSONL logs for replay and threshold tuning before enabling any bounded command output.
